# EGFR pIC50 — EDA & Chemical Space

**Dataset:** `CHEMBL203` curated IC50 → pIC50 (10,502 molecules after curation)  
**Goal:** Understand the distribution of activity, molecular properties, and chemical diversity before modelling.

## Sections
1. Dataset overview
2. pIC50 distribution & activity class split
3. Molecular descriptor distributions (RDKit)
4. Lipinski Rule of Five compliance
5. Chemical space — UMAP of Morgan fingerprints
6. Scaffold diversity (Murcko)
7. pIC50 vs descriptor correlations

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('tab10')

## 1. Load data

In [ ]:
parquet_path = Path('../data/processed/CHEMBL203_curated.parquet')
csv_path     = Path('../data/curated_dataset.csv')

if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
    df = df.rename(columns={'pic50': 'pIC50', 'canonical_smiles': 'smiles'})
else:
    df = pd.read_csv(csv_path, index_col=0)

print(f"Loaded {len(df):,} records — columns: {df.columns.tolist()}")
df.head()

## 2. Dataset overview

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\npIC50 summary:")
df['pIC50'].describe().round(3)

## 3. pIC50 distribution & activity class split

A common threshold for EGFR: **pIC50 ≥ 6** (IC50 ≤ 1 µM) = active, **< 6** = inactive.  
This threshold matters for scaffold-split class balance checks before model training.

In [ ]:
ACTIVITY_THRESHOLD = 6.0

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- histogram ---
axes[0].hist(df['pIC50'], bins=60, color='steelblue', edgecolor='white')
axes[0].axvline(df['pIC50'].median(), color='red',    linestyle='--', lw=1.5,
                label=f'Median {df["pIC50"].median():.2f}')
axes[0].axvline(df['pIC50'].mean(),   color='orange', linestyle='--', lw=1.5,
                label=f'Mean {df["pIC50"].mean():.2f}')
axes[0].axvline(ACTIVITY_THRESHOLD,   color='black',  linestyle=':',  lw=1.5,
                label=f'Threshold {ACTIVITY_THRESHOLD}')
axes[0].set_xlabel('pIC50')
axes[0].set_ylabel('Count')
axes[0].set_title('pIC50 Distribution')
axes[0].legend(fontsize=9)

# --- activity class bar ---
n_active   = (df['pIC50'] >= ACTIVITY_THRESHOLD).sum()
n_inactive = len(df) - n_active
bars = axes[1].bar(['Active\n(pIC50 ≥ 6)', 'Inactive\n(pIC50 < 6)'],
                   [n_active, n_inactive], color=['steelblue', 'salmon'])
for bar, count in zip(bars, [n_active, n_inactive]):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 80,
                 f'{count:,}\n({count/len(df)*100:.1f}%)', ha='center', fontsize=10)
axes[1].set_ylabel('Count')
axes[1].set_title(f'Activity Class Split (threshold = {ACTIVITY_THRESHOLD})')

plt.tight_layout()
plt.show()

print(f"Active   (pIC50 >= {ACTIVITY_THRESHOLD}): {n_active:,}  ({n_active/len(df)*100:.1f}%)")
print(f"Inactive (pIC50 <  {ACTIVITY_THRESHOLD}): {n_inactive:,}  ({n_inactive/len(df)*100:.1f}%)")

## 4. Molecular descriptors

Computing eight core RDKit descriptors used in Lipinski-style drug-likeness analysis.

In [ ]:
def compute_descriptors(smiles: str) -> dict | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {
        'MW':       Descriptors.MolWt(mol),
        'LogP':     Descriptors.MolLogP(mol),
        'HBD':      rdMolDescriptors.CalcNumHBD(mol),
        'HBA':      rdMolDescriptors.CalcNumHBA(mol),
        'TPSA':     rdMolDescriptors.CalcTPSA(mol),
        'RotBonds': rdMolDescriptors.CalcNumRotatableBonds(mol),
        'Rings':    rdMolDescriptors.CalcNumRings(mol),
        'ArRings':  rdMolDescriptors.CalcNumAromaticRings(mol),
    }

desc_df = pd.DataFrame(df['smiles'].apply(compute_descriptors).tolist())
df_full = pd.concat([df.reset_index(drop=True), desc_df], axis=1)
print(f"Descriptors computed for {desc_df.notna().all(axis=1).sum():,} / {len(df):,} molecules")
desc_df.describe().round(2)

In [ ]:
descriptors = ['MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotBonds', 'Rings', 'ArRings']
colors = sns.color_palette('tab10', len(descriptors))

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col, color in zip(axes.flat, descriptors, colors):
    ax.hist(df_full[col].dropna(), bins=40, color=color, edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel('Count')

plt.suptitle('RDKit Molecular Descriptor Distributions', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## 5. Lipinski Rule of Five compliance

Oral bioavailability heuristic: MW ≤ 500, LogP ≤ 5, HBD ≤ 5, HBA ≤ 10.  
EGFR inhibitors (especially 3rd-gen covalent ones like osimertinib) sometimes breach MW/LogP.

In [ ]:
rules = {
    'MW ≤ 500':  df_full['MW']  <= 500,
    'LogP ≤ 5':  df_full['LogP'] <= 5,
    'HBD ≤ 5':   df_full['HBD']  <= 5,
    'HBA ≤ 10':  df_full['HBA']  <= 10,
}
all_pass = pd.concat(rules.values(), axis=1).all(axis=1)

print(f"{'Rule':<15} {'Pass':>6}  {'%':>6}")
print("-" * 30)
for rule, mask in rules.items():
    n = mask.sum()
    print(f"{rule:<15} {n:>6}  {n/len(df_full)*100:>5.1f}%")
print("-" * 30)
n = all_pass.sum()
print(f"{'All 4 rules':<15} {n:>6}  {n/len(df_full)*100:>5.1f}%")

# violin: Ro5-compliant vs violators, split by pIC50
df_full['Ro5'] = all_pass.map({True: 'Compliant', False: 'Violator'})
fig, ax = plt.subplots(figsize=(7, 4))
sns.violinplot(data=df_full, x='Ro5', y='pIC50', palette=['steelblue', 'salmon'],
               inner='quartile', ax=ax)
ax.set_title('pIC50 distribution: Lipinski-compliant vs violators')
plt.tight_layout()
plt.show()

## 6. Chemical space — UMAP of Morgan fingerprints

Morgan fingerprints (ECFP4, radius=2, 2048 bits) compressed to 2D with UMAP using Jaccard distance.  
Colour = pIC50. Reveals activity clusters and whether potent compounds are chemically diverse or concentrated.

In [ ]:
fps, valid_idx = [], []
for i, smi in enumerate(df_full['smiles']):
    mol = Chem.MolFromSmiles(smi)
    if mol:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
        arr = np.zeros(2048, dtype=np.uint8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        fps.append(arr)
        valid_idx.append(i)

X = np.array(fps)
print(f"Fingerprints ready: {X.shape}  (n_molecules × bits)")

In [ ]:
from umap import UMAP

print("Running UMAP (Jaccard, n_neighbors=30) — takes ~1–2 min for 10k molecules…")
reducer = UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
               metric='jaccard', random_state=42, verbose=False)
embedding = reducer.fit_transform(X)

df_umap = df_full.iloc[valid_idx].copy().reset_index(drop=True)
df_umap['umap_x'] = embedding[:, 0]
df_umap['umap_y'] = embedding[:, 1]
print("Done.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(
    df_umap['umap_x'], df_umap['umap_y'],
    c=df_umap['pIC50'], cmap='RdYlGn',
    s=3, alpha=0.6, rasterized=True
)
plt.colorbar(sc, ax=ax, label='pIC50')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_title('EGFR Chemical Space — ECFP4 UMAP (coloured by pIC50)')
plt.tight_layout()
plt.show()

## 7. Scaffold diversity (Murcko)

Murcko scaffolds strip all side chains, leaving the ring system.  
High singleton fraction = chemically diverse dataset → scaffold split will be a meaningful generalization test.

In [ ]:
def get_murcko(smiles: str) -> str | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    scaf = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaf)

df_full['scaffold'] = df_full['smiles'].apply(get_murcko)

scaffold_counts = df_full['scaffold'].value_counts()
n_total    = df_full['scaffold'].notna().sum()
n_unique   = scaffold_counts.shape[0]
n_singleton = (scaffold_counts == 1).sum()

print(f"Total molecules with scaffolds : {n_total:,}")
print(f"Unique Murcko scaffolds        : {n_unique:,}  ({n_unique/n_total*100:.1f}% diversity)")
print(f"Singleton scaffolds            : {n_singleton:,}  ({n_singleton/n_unique*100:.1f}% of all scaffolds)")
print(f"\nTop 10 most common scaffolds:")
print(scaffold_counts.head(10).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(scaffold_counts.values, bins=60, color='steelblue', edgecolor='white', log=True)
ax.set_xlabel('Molecules per scaffold')
ax.set_ylabel('Number of scaffolds (log scale)')
ax.set_title('Scaffold Frequency Distribution (Murcko)')
plt.tight_layout()
plt.show()

## 8. pIC50 vs descriptor correlations

Which descriptors correlate most with potency?  
Strong correlations can inform feature selection and explain model behaviour.

In [ ]:
from scipy.stats import spearmanr

print(f"{'Descriptor':<12} {'Spearman ρ':>12}  {'p-value':>10}")
print("-" * 38)
for col in descriptors:
    valid = df_full[['pIC50', col]].dropna()
    rho, pval = spearmanr(valid['pIC50'], valid[col])
    print(f"{col:<12} {rho:>12.4f}  {pval:>10.2e}")

In [ ]:
corr_cols = descriptors + ['pIC50']
corr_matrix = df_full[corr_cols].corr(method='spearman')

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, square=True, linewidths=0.5, ax=ax
)
ax.set_title('Spearman Correlation — Descriptors vs pIC50')
plt.tight_layout()
plt.show()